In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
from churn.dbx import get_config
from churn import transforms as T
from delta.tables import DeltaTable
from pyspark.sql import functions as F

cfg = get_config(dbutils)

In [0]:
def upsert(df, table, keys, cluster_by=None, update=False):
    """Insère les nouvelles lignes dans une table Delta, sans créer de doublons."""
    if not spark.catalog.tableExists(table):            # 1re fois : on crée la table
        w = df.write
        if cluster_by:
            w = w.clusterBy(*cluster_by)                 # Liquid Clustering : lectures par client plus rapides
        w.saveAsTable(table)
        return
    # Fois suivantes : MERGE = "si la clé existe déjà, (mettre à jour) ; sinon, insérer"
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)  # ex. "t.operation_id = s.operation_id"
    m = DeltaTable.forName(spark, table).alias("t").merge(df.alias("s"), cond)
    if update:
        m = m.whenMatchedUpdateAll()                     # ex. un client qui change de segment
    m.whenNotMatchedInsertAll().execute()

In [0]:
valid_ops, rejected_ops = T.clean_operations(spark.table(cfg.t("bronze_operations")))
upsert(valid_ops, cfg.t("silver_operations"), ["operation_id"], cluster_by=["client_id"])
(rejected_ops.withColumn("_rejected_at", F.current_timestamp())
 .write.mode("overwrite").saveAsTable(cfg.t("quarantine_operations")))

upsert(T.clean_interactions(spark.table(cfg.t("bronze_interactions"))),
       cfg.t("silver_interactions"), ["interaction_id"], cluster_by=["client_id"])
upsert(T.clean_clients(spark.table(cfg.t("bronze_clients"))),
       cfg.t("silver_clients"), ["client_id"], update=True)
upsert(T.clean_clotures(spark.table(cfg.t("bronze_clotures"))),
       cfg.t("silver_clotures"), ["client_id"], update=True)

In [0]:
display(spark.sql(f"""
  SELECT 'silver_operations' AS table_name, COUNT(*) AS n FROM {cfg.t('silver_operations')}
  UNION ALL SELECT 'quarantine_operations', COUNT(*) FROM {cfg.t('quarantine_operations')}
  UNION ALL SELECT 'silver_interactions', COUNT(*) FROM {cfg.t('silver_interactions')}
  UNION ALL SELECT 'silver_clients', COUNT(*) FROM {cfg.t('silver_clients')}
  UNION ALL SELECT 'silver_clotures', COUNT(*) FROM {cfg.t('silver_clotures')}"""))

display(spark.table(cfg.t("quarantine_operations")).groupBy("_failed_rules").count())